# Vector + Graph Retriever Agent

You will modify the agent to include an additional tool that:

1. Searches the documents using the vector index
2. Traverses the graph around the document to find other facts

---

## 1. Configuration

In [ ]:
#################################################
# CONFIGURATION
#################################################

# AWS Bedrock Configuration
INFERENCE_PROFILE_ARN = "PASTE_YOUR_ARN_HERE"  # <-- PASTE HERE
REGION = "us-west-2"

# OpenAI Configuration (for embeddings)
# The vector index was created with OpenAI text-embedding-ada-002
OPENAI_API_KEY = "PASTE_YOUR_OPENAI_API_KEY_HERE"  # <-- PASTE HERE

# Neo4j Configuration
NEO4J_URI = "neo4j+s://your-instance.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "your-password"

#################################################

# Validate configuration
errors = []
if "PASTE" in INFERENCE_PROFILE_ARN or "YOUR" in INFERENCE_PROFILE_ARN:
    errors.append("Paste your inference profile ARN (run ./setup-inference-profile.sh)")
if "PASTE" in OPENAI_API_KEY or "YOUR" in OPENAI_API_KEY:
    errors.append("Paste your OpenAI API key")
if "your-instance" in NEO4J_URI:
    errors.append("Update NEO4J_URI with your Neo4j Aura connection string")
if "your-password" in NEO4J_PASSWORD:
    errors.append("Update NEO4J_PASSWORD with your Neo4j password")

if errors:
    print("ERROR: Configuration incomplete!")
    for e in errors:
        print(f"  - {e}")
else:
    print("Configuration OK!")

## 2. Setup

Load the environment variables, import the required modules, connect to the Neo4j database, and set up configuration.

In [ ]:
# Install missing packages
%pip install langgraph langchain-aws neo4j-graphrag boto3 openai -q

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.embeddings import OpenAIEmbeddings

from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports successful!")

In [ ]:
# Connect to Neo4j
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## 3. Create Embedding Model

To use the vector index, you will need to create an embedding model to convert user queries into embeddings.

The vector index was created with OpenAI `text-embedding-ada-002`, so we use the same model for queries.

In [ ]:
# Create the embedding model using OpenAI (matching the vector index)
embedder = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-ada-002",
)

print("Embedder initialized with model: text-embedding-ada-002 (OpenAI)")

## 4. Create Retrieval Query

To retrieve data from the graph after documents have been found, you can define a `retrieval_query`.

This `retrieval_query` is appended to the Cypher query automatically generated by the neo4j-graphrag-python library's `VectorCypherRetriever`. The library first calls the vector index and yields `node` and `score` variables, which are then used by this query.

The conceptual flow is:

```cypher
CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
YIELD node, score
// ... then this retrieval_query is appended here ...
```

*   `node`: This represents the specific Chunk node found by the vector search (the text segment that mathematically matches your query).
*   `score`: The similarity score (0.0 to 1.0) of the vector match.

In [ ]:
retrieval_query = """
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, score, company, collect(risk.name)[0..20] AS risks
WHERE score IS NOT NULL
RETURN 
    node.text AS text,
    score,
    {company: company.name, risks: risks} AS metadata
ORDER BY score DESC
"""

print("Retrieval query defined!")

> This query retrieves the `Company` the `Document` relates to and any associated `RiskFactor` nodes.

---

## 5. Create Vector Retriever

In [ ]:
# Create vector retriever with graph context
vector_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    retrieval_query=retrieval_query,
)

print("Vector retriever created!")

## 6. Define Tools

Create the tools for the agent. In LangGraph, tools are Python functions with the `@tool` decorator:
- A descriptive docstring (used by the agent to decide when to use the tool)
- Type hints for parameters

In [ ]:
@tool
def get_graph_schema() -> str:
    """Get the schema of the graph database including node labels, relationships, and properties."""
    return get_schema(driver)


@tool
def retrieve_financial_documents(query: str) -> str:
    """Find details about companies in their financial documents using semantic search.
    
    Args:
        query: The search query to find relevant documents
    """
    try:
        results = vector_retriever.search(query_text=query, top_k=3)
        if not results.items:
            return "No documents found matching the query."
        return "\n\n".join(item.content for item in results.items)
    except Exception as e:
        return f"Error searching documents: {e}"


# Add the tools to a list
tools = [get_graph_schema, retrieve_financial_documents]

print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

> The agent will use the tool's name and docstring to determine if it is needed.

---

## 7. Create Agent

In [ ]:
# Initialize LLM using AWS Bedrock
llm = ChatBedrockConverse(
    model=INFERENCE_PROFILE_ARN,
    provider="anthropic",
    region_name=REGION,
    temperature=0,
)

# Create the agent with tools
SYSTEM_PROMPT = """You are a helpful assistant that can answer questions about 
a graph database containing financial documents. You can retrieve 
the schema and search for relevant documents.

Use the get_graph_schema tool to understand the database structure.
Use the retrieve_financial_documents tool to search for information in company filings.

Be concise and informative in your responses."""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("Agent created with 2 tools!")

> The agent has access to the `get_graph_schema` and `retrieve_financial_documents` tools. The agent will pick between them when processing the user's query.

---

## 8. Run the Agent

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"User: {question}")
    print("-" * 50)
    
    result = agent.invoke({"messages": [("human", question)]})
    
    # Get the final message
    final_message = result["messages"][-1]
    print(f"\nAssistant: {final_message.content}")
    return result

In [ ]:
query = "Summarise what risk factors are mentioned in Apple's financial documents?"
result = run_agent(query)

## 9. Experiment

Experiment with the agent, ask different questions about the documents and the graph schema, for example:

* Summarize the schema of the graph database.
* What are the main risk factors mentioned in the documents?
* Tell me about cybersecurity threats in financial services
* What products does Microsoft mention in its financial documents?
* How are companies connected through their mentioned products?
* What type of questions can I ask about Apple using the graph database?

> The agent will pick different tools depending on the task.

---

Try modifying the `retrieval_query` to pull back additional data about the `Company` such as:

* Asset managers - `(company:Company)<-[:OWNS]-(manager:AssetManager)`
* Financial metrics - `(company:Company)-[:HAS_METRIC]->(metric:FinancialMetric)`
* Products - `(company:Company)-[:MENTIONS]->(product:Product)`

Including additional context will help the agent to create more specific responses.

In [ ]:
# Try: Schema question (will use get_graph_schema)
query = "Summarize the schema of the graph database."
result = run_agent(query)

In [ ]:
# Try: Document search question (will use retrieve_financial_documents)
query = "What products does Microsoft mention in its financial documents?"
result = run_agent(query)

In [ ]:
# Try: Another document search
query = "Tell me about cybersecurity threats mentioned in financial filings."
result = run_agent(query)

---

[Move on to the Multi-Tool Agent with Text2Cypher Notebook](03_text2cypher_agent.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")